In [ ]:
import duckdb
import pandas as pd
from pathlib import Path
import diskcache
from itertools import chain

from utils.pandas_setup import pandas_setup
pandas_setup()

import contextlib
from unidecode import unidecode
from nameparser import HumanName

import pyalex
from pyalex import Works, Authors, Sources, Institutions, Topics, Publishers, Funders
pyalex.config.email = "Lawrence.Cram@anu.edu.au"
pyalex.config.max_retries = 0
pyalex.config.retry_backoff_factor = 0.1
pyalex.config.retry_http_codes = [429, 500, 503]

MY_DATA_PATH = Path('../DATA/')
MY_DATABASE_FILE = Path(MY_DATA_PATH / 'econ.duckdb')
MY_CACHE_FILE = Path('/home/lc/m/.cache/econommicsbusiness/cache.db')
DATAFILES_PATH = Path('../DATAFILES')

def normalise_name(in_name: str=None) -> str:
    in_name = ' '.join([part.strip() for part in unidecode(in_name).split(' ')])
    in_name = in_name.title()
    name = HumanName(in_name)
    # if name.title == 'Md.Shahriar':
    #     name.first = 'Shahriar'
    #     name.title = ''
    #     # print(f'{name = }')
    # if name.title == 'Mahdi':
    #     name.first = 'Mahdi'
    #     name.title = ''
    #     # print(f'{name = }')
    # print(f'{name = }')
    if name.middle:
        return f'{name.first} {name.middle} {name.last}'
    return f'{name.first} {name.last}'

In [8]:
class SetUp:

    def __init__(self):
        self._setup_db()
        return

    def _setup_db(self):
        self.db = duckdb.connect(MY_DATABASE_FILE)
        self.db.sql("ATTACH IF NOT EXISTS ':memory:'")
        self.db.sql(""" SET memory_limit = '56GB';
                        SET threads = 6;
                        SET preserve_insertion_order = false;
                        SET order_by_non_integer_literal=true;
                        SET enable_progress_bar = true;
                        SET temp_directory = '/home/lc/m/.tmp';
                    """)
        self.db.sql("SHOW ALL TABLES").show()
    
        with contextlib.suppress(Exception):
            self.db.create_function('normalise_name', 
                                        normalise_name, 
                                        return_type=duckdb.duckdb.typing.DuckDBPyType(str), 
                                        exception_handling='return_null',
                                        null_handling='special',
                                        side_effects=True
                                    )
            # self.db.create_function('extract_works', 
            #                             extract_works, 
            #                             return_type=duckdb.duckdb.typing.DuckDBPyType(dict[str, str]), 
            #                         )
        self.db.sql("SHOW ALL TABLES").show()
        return

In [9]:
class CorpusETL(SetUp):

    def __init__(self):
        super().__init__()
        return
    
    def extract_corpus(self, author_id=None):
        print(f'{author_id = }')
        print(Authors()[author_id])
        return
    
    def extract_corpus_endogenous(self, author_id=None):
        pick = f"WHERE author_id='{author_id}'" if author_id else ""
        sql = f"""
                SELECT *
                    FROM works w
                    RIGHT JOIN 
                        (SELECT * 
                            FROM authorships 
                            {pick}
                        )
                        USING (work_id)
                    WHERE contains('article review preprint letter', w.type) = true
            """
        df = self.db.sql(sql).df()
        print(df)
        return

In [10]:
    
class MatchDomingoSample(SetUp):

    def __init__(self):
        super().__init__()
        return    

    def extract_sample(self):
            sample = pd.read_excel('../RESULTS/researchers_results.xlsx').drop(columns=['Unnamed: 0', 'NAME'])
            print(f'{sample.shape = }\n{sample.head()}')
            sample['Research_Profile'] = [normalise_name(f"{n.split(', ')[1]} {n.split(', ')[0]}") if ',' in n else normalise_name(n) for n in sample.Research_Profile]
            sample['first'] = [normalise_name(n).split(' ', maxsplit=1)[0] for n in sample.Research_Profile]
            sample['last'] = [normalise_name(n).rsplit(' ', maxsplit=1)[1] for n in sample.Research_Profile]
            sample = sample.sort_values('last').reset_index(drop=True)
            print(f'{sample.shape = }\n{sample.head(256)}')
            self.db.sql("CREATE OR REPLACE TABLE econ.samples AS SELECT * FROM sample")
            self.db.sql("SELECT * FROM econ.samples").show()
            return
    
    def match_sample(self):
        sql = """ 
            SELECT Research_Profile,
                    author_id,
                    author_name,
                    "Group", 
                    CIT,
                    PUB,
                    HCP,
                    regexp_extract(a.author_name, ' \\w+$') AS last_all,
                    s.last AS last_sample,
                    count(work_id) AS works_count
                FROM econ.samples s
                    LEFT JOIN authorships a
                        ON normalise_name(a.author_name) = s.Research_Profile 
                            OR regexp_extract(a.author_name, ' \\w+$') = s.last
                    --WHERE a.author_name NOT NULL
                    GROUP BY ALL
                    ORDER BY s.last ASC, works_count DESC
            """
        df = self.db.sql(sql).df()
        # df = df.drop_duplicates(subset='author_name')
        print(f'{df.shape = }\n{df.head(256)}')
        self.db.sql("CREATE OR REPLACE TABLE econ.sample_names AS SELECT * FROM df")
        return


In [11]:
def main():

    # cetl = CorpusETL()
    # cetl.extract_corpus_endogenous() #(author_id='https://openalex.org/A5101600363')

    mds = MatchDomingoSample()
    mds.extract_sample()
    mds.match_sample()

    return

In [ ]:
if __name__ == "__main__":
    main()
    print("DONE!")

┌──────────┬─────────┬────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────